# Activity 2: Text AI, Two Clouds, Two Different Answers

**Week 6 Day 4 | AWS Comprehend vs. GCP Natural Language on real messages**

**Estimated time:** 75 minutes
**Difficulty:** Beginner to Intermediate
**Format:** Individual
**Prerequisites:** [Activity 1](./Activity_1_Vision_AI_AWS_vs_GCP.ipynb) complete, credentials working

## The job

Activity 1 turned pixels into a table. This one turns paragraphs into a table.

You have a pile of messages. Somebody wants to know how customers feel, at scale, without reading all of them. Two clouds sell a managed model that will label sentiment for you, one API call per message, no training.

Here is the twist that makes this more interesting than Activity 1: **the two providers do not return the same kind of answer.** AWS gives you a word. Google gives you two numbers. You cannot put them in the same column until you decide, yourself, what those numbers mean. That decision is not a technicality. It is the most consequential line of code in the notebook, and there is no documentation page that will make it for you.

## How this notebook works

Same as Activity 1. Every idea is explained, then worked once on a single message in small inspectable cells, then handed to you as a `TODO` for the batch. Run the worked cells and read the output before you write the `TODO` below them.

## What you will learn

- How to navigate an unfamiliar API response and find the part you need
- Why a confidence score is often more informative than the label sitting on top of it
- What `MIXED` is, why it is not `NEUTRAL`, and why it will break your comparison
- How to read the docs for limits and for operations you did not know existed
- How to build a defensible translation layer between two incompatible vendor outputs
- Why the same code produces 62 percent agreement on one corpus and 83 percent on another

---
## Setup

If either check prints `False`, go back to [Activity 0](./Activity_0_Environment_and_API_Setup.md).

In [ ]:
import os
import json

import boto3
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

comprehend = boto3.client("comprehend")

print("AWS credentials loaded:", "AWS_ACCESS_KEY_ID" in os.environ)
print("GCP_API_KEY loaded:    ", "GCP_API_KEY" in os.environ)

---
# 1. Comprehend is a service, not a function

Same mental model as Rekognition. "Sentiment analysis" is one item on a menu of pre-trained NLP models, and the client object will show you the whole menu.

In [ ]:
[m for m in dir(comprehend) if m.startswith(("detect_", "batch_detect_"))]

| Operation | What it answers |
| :--- | :--- |
| `detect_sentiment` | Is this text positive, negative, neutral, or mixed |
| `detect_targeted_sentiment` | Sentiment **per entity**, so one review can be positive about the food and negative about the wait |
| `detect_entities` | Which real-world things are named here (people, organizations, dates, quantities) |
| `detect_key_phrases` | Which noun phrases carry the content |
| `detect_pii_entities` | Where is the personally identifiable information, so you can redact it |
| `detect_dominant_language` | What language is this, and how sure are we |
| `detect_toxic_content` | Does this text contain harassment, hate speech, threats, or profanity |
| `detect_syntax` | Part-of-speech tagging |
| `batch_detect_*` | The same operations, several documents per call |

Two of those deserve a flag now, because they solve problems people usually assume require an LLM:

- `detect_pii_entities` finds names, addresses, account numbers, and SSNs with character offsets. If your job is "scrub the claim notes before they leave the building," this is a single call, not a prompt you have to keep testing.
- `detect_targeted_sentiment` is the honest answer to "but this message is positive about one thing and negative about another." Document-level sentiment flattens that. Entity-level sentiment does not.

**Reference:** [Comprehend API operations](https://docs.aws.amazon.com/comprehend/latest/APIReference/API_Operations.html)

---
# 2. The data: real messages, written by real people

`social_posts_sample.csv` is a sample of real public social posts. Not written for a lesson, not cleaned up, not balanced. Load it and look before you send a single character to an API.

In [ ]:
raw_posts = pd.read_csv("social_posts_sample.csv")

print(raw_posts.shape)
raw_posts.head()

In [ ]:
raw_posts.columns.tolist()

In [ ]:
raw_posts.info()

Three things to notice before going further, because each one will bite something later:

1. **The column names have spaces in them.** `Time of Tweet`, `Age of User`, `Land Area (Km²)`. You cannot write `raw_posts.Time of Tweet`. You have to use `raw_posts["Time of Tweet"]`. Real files are full of this.
2. **There is no sentiment label.** Nothing in this file tells you the right answer. That means you cannot compute accuracy today, only agreement, and you should be suspicious of anyone who blurs that line.
3. **The text is genuinely messy.** Read a few in full.

In [ ]:
for text in raw_posts["text"].head(8):
    print(repr(text))
    print()

Backtick apostrophes instead of real ones (`don\`t`). Censored profanity as asterisks. Typos. Fragments with no verb. All caps. Missing context, because half of these are replies to something you cannot see.

This is what text looks like in the wild, and it is why the results later in this notebook are messier than a tutorial would lead you to expect. That messiness is the finding, not a flaw in the exercise.

---
# 3. Develop on a slice, not the whole file

You are about to call two paid APIs once per row. Do not point that at the full file while you are still writing the loop.

Take a working slice. Get the pipeline correct on 60 rows, then scale. Every iteration on a bug costs you the whole batch again otherwise, in both time and money, and you will iterate more than you think.

In [ ]:
posts = raw_posts.head(60).copy()

print(f"working with {len(posts)} of {len(raw_posts)} rows")

---
# 4. One call, and the whole response

Pick the first message and read it yourself before the model does. Form an opinion. You are going to want it in about three cells.

In [ ]:
example_text = posts["text"].iloc[0]

print(example_text)

Is that positive, negative, or neutral? Commit to an answer before you scroll.

In [ ]:
sentiment_response = comprehend.detect_sentiment(Text=example_text, LanguageCode="en")

sentiment_response

As in Activity 1, this is a plain dictionary with the useful part and the plumbing mixed together.

- `Sentiment` is the label. One string.
- `SentimentScore` is a dictionary of four confidence values.
- `ResponseMetadata` is HTTP bookkeeping.

Pull the label out first.

In [ ]:
sentiment_response["Sentiment"]

In [ ]:
for label, score in sentiment_response["SentimentScore"].items():
    print(f"{label:>9}: {score:.4f}")

### Stop and look at those four numbers

This is the most important cell in the notebook.

The API handed you a confident-sounding one-word label. The scores underneath it tell a completely different story: when this notebook was written, `Positive` and `Neutral` came back within about three thousandths of each other. The model was essentially flipping a coin, and the label reported the winner of that coin flip with no indication that it was close.

If you had only taken `response["Sentiment"]` and moved on, which is exactly what a tutorial would have you do, you would have written a near-tie into a database column as a fact.

**This generalizes well past sentiment.** Any classifier that returns a label is returning `argmax` over a distribution, and `argmax` is lossy by construction. The scores are where the uncertainty lives. Keep them.

Two things follow, and you will implement both later in this notebook:

- **Store the scores, not just the label.** They cost nothing extra, they are already in the response, and you cannot reconstruct them later without paying for every call again.
- **Gate on the winning score.** A row where the top score is 0.44 is not the same kind of row as one where it is 0.99, and a pipeline that treats them identically is throwing away the only signal it has about which rows a human should look at.

---
# 5. The fourth label nobody expects

Most people assume sentiment has three values. Comprehend has four, and the fourth one will quietly break your comparison later if you have not met it.

In [ ]:
probe = comprehend.detect_sentiment(
    Text="It's an okay movie, not great but not bad either.",
    LanguageCode="en",
)

print("label:", probe["Sentiment"])
for label, score in probe["SentimentScore"].items():
    print(f"{label:>9}: {score:.4f}")

`MIXED` is not a tie between positive and negative, and it is not a synonym for neutral. The distinction is real:

- **`NEUTRAL`** means there is no emotional content. "Requesting a copy of my declarations page." Nothing is being felt.
- **`MIXED`** means there is emotional content pulling in **both** directions in the same text. "The adjuster was wonderful but the payout was insulting." Two feelings, both strong, opposite signs.

Averaging them into one number destroys the difference, which is exactly what you are about to be forced to do, because Google's API only gives you one number. Remember that this label exists when your agreement rate comes back lower than you expected.

---
# 6. Read the docs

You have seen what one call returns. You still do not know what the service refuses, what it caps, or what cheaper option you are not using.

Answer these from the documentation. Do not guess, and do not ask an AI assistant.

**Reference:** [`DetectSentiment` API reference](https://docs.aws.amazon.com/comprehend/latest/APIReference/API_DetectSentiment.html)

### Question 1
What is the maximum size of the `Text` you can send in one call? What exception comes back if you exceed it? Some of Hartford Mutual's claim adjuster notes run several pages. What would you have to do first?

### Question 2
What are the exact valid values of `Sentiment`? Confirm the list from the docs rather than from the four calls you have made.

### Question 3
You have 400,000 messages to classify. Making 400,000 separate HTTPS calls is slow. Find the operation that sends more than one document per call. How many can you send at once, and what does its response look like when one document in the batch fails?

**Reference for Question 3:** [`BatchDetectSentiment`](https://docs.aws.amazon.com/comprehend/latest/APIReference/API_BatchDetectSentiment.html)

**Your answers:**

1. Maximum text size, exception, and what to do with a long document:
2. Valid `Sentiment` values:
3. Batch operation, batch size, and failure behavior:

---
# 7. Sentiment is not the only question you can ask

Before scaling, look at two other operations on a message where they actually have something to find. The claims file has one written by a long-time customer.

In [ ]:
claims = pd.read_csv("claims_feedback_sample.csv")

claims_example = claims[claims["message_id"] == "MSG_017"]["text"].iloc[0]
print(claims_example)

In [ ]:
entity_response = comprehend.detect_entities(Text=claims_example, LanguageCode="en")

for entity in entity_response["Entities"]:
    print(f"{entity['Type']:>14}  {entity['Text']!r:35}  score={entity['Score']:.3f}  "
          f"chars {entity['BeginOffset']}-{entity['EndOffset']}")

Entities are typed spans of text, and the type vocabulary is fixed: `PERSON`, `LOCATION`, `ORGANIZATION`, `COMMERCIAL_ITEM`, `EVENT`, `DATE`, `QUANTITY`, `TITLE`, `OTHER`.

The `BeginOffset` and `EndOffset` are why this is a genuinely useful operation rather than a curiosity. They are character positions into your original string, so you can slice the exact span back out. That is what makes redaction possible: find the span, replace the span, keep the rest of the message intact.

In [ ]:
# Offsets point back into the original string. Slice one out to prove it.
first = entity_response["Entities"][0]

print(repr(claims_example[first["BeginOffset"]:first["EndOffset"]]))

In [ ]:
key_phrase_response = comprehend.detect_key_phrases(Text=claims_example, LanguageCode="en")

for phrase in key_phrase_response["KeyPhrases"]:
    print(f"{phrase['Score']:.3f}  {phrase['Text']!r}")

Compare the two lists. They overlap but they are not the same job.

**Entities** answer "which real-world things are named here," and each one is tagged with a type you can filter on. **Key phrases** answer "what is this text about," with no typing at all. If you want every message that mentions an organization, you need entities. If you want a tag cloud, key phrases are cheaper to work with.

Picking the wrong one is a common and expensive mistake, because both return plausible-looking output and neither one errors.

---
# 8. Scale AWS across the batch

You have the pattern. Now run it over all 60 posts.

**Keep the scores.** Section 4 was about exactly this. One row per message, with the label and all four confidence values, because you cannot get them back later without paying again.

In [ ]:
# TODO: Run detect_sentiment over every row in `posts` and build aws_rows.
#
# For each row, append a dict with:
#   "textID"            -> row["textID"]
#   "text"              -> row["text"]
#   "aws_sentiment"     -> the label
#   "aws_positive"      -> SentimentScore["Positive"]
#   "aws_negative"      -> SentimentScore["Negative"]
#   "aws_neutral"       -> SentimentScore["Neutral"]
#   "aws_mixed"         -> SentimentScore["Mixed"]
#
# Hint, iterating a DataFrame:  for _, row in posts.iterrows():
# Hint, unpacking the scores:   scores = response["SentimentScore"]
#                               then scores["Positive"], and so on.
# Hint, failure isolation: same try/except pattern as Activity 1. One bad row
#       must not cost you the 40 calls you already paid for.
# Hint: text must be a non-empty string. str(row["text"]).strip() guards against
#       a stray NaN or a blank row, which the API rejects.
#
# 60 calls, roughly 20 to 40 seconds.

aws_rows = []

aws_df = None

You should see all four labels represented, with `MIXED` appearing on only a handful of rows. That handful is going to matter in Section 13.

In [ ]:
# TODO: Add two columns that turn the raw scores into something actionable.
#
#   "aws_confidence" -> the highest of the four scores for that row.
#                       This is how sure the model was about the label it chose.
#   "needs_review"   -> True when aws_confidence is below 0.60.
#
# Then print how many rows fall below the line, and look at a few of them.
#
# Hint: aws_df[["aws_positive", "aws_negative", "aws_neutral", "aws_mixed"]].max(axis=1)
#       axis=1 takes the max ACROSS the four columns for each row,
#       which is what you want. axis=0 would give you the max down each column.

Read the flagged messages. They should mostly be short, contextless, or genuinely ambiguous, which is the point: the confidence score found the hard rows without you writing a single rule about what "hard" means.

That is the whole design of a human-in-the-loop pipeline. Auto-accept the confident rows, route the rest to a person, and use the model's own uncertainty to draw the line. The threshold is a business decision, not a technical one, and Section 14 makes you defend one.

---
# 9. GCP Natural Language, one call first

Same REST pattern as Activity 1, different endpoint:

```
POST https://language.googleapis.com/v1/documents:analyzeSentiment?key=YOUR_API_KEY
```

```json
{ "document": { "type": "PLAIN_TEXT", "content": "your text here" } }
```

`document` is an object rather than a bare string because `type` can also be `HTML`, in which case Google strips the markup for you instead of scoring your `<div>` tags. There is also an optional `encodingType`, which controls how the character offsets in the response are counted.

Run it on the exact same message AWS scored in Section 4.

In [ ]:
gcp_response = requests.post(
    "https://language.googleapis.com/v1/documents:analyzeSentiment",
    params={"key": os.environ["GCP_API_KEY"]},
    json={"document": {"type": "PLAIN_TEXT", "content": example_text}},
)

print("HTTP status:", gcp_response.status_code)
print(json.dumps(gcp_response.json(), indent=1))

Look at what came back and compare it to the AWS response, side by side:

| | AWS Comprehend | GCP Natural Language |
| :--- | :--- | :--- |
| The answer | `Sentiment`, one of four words | `documentSentiment`, two floats |
| Confidence | Four scores summing to 1 | No confidence score at all |
| Per sentence | Not offered | `sentences`, each with its own score |
| Language | Assert it in the request | Detected and returned |

Three genuine differences hide in that table.

**Google gives you no confidence.** Everything Section 4 taught you about keeping the scores has no equivalent here. There is no "how sure are you" number to gate on. You can approximate one with `magnitude`, but that is your inference, not the provider's.

**Google gives you per sentence scores for free.** `sentences` breaks the document down without a second call. AWS charges you a separate `detect_targeted_sentiment` call for anything comparable.

**Google detects the language.** AWS makes you declare `LanguageCode="en"` and will happily score French text as though it were English if you lie to it.

### `score` and `magnitude` are answering different questions

This trips up nearly everyone the first time, so be precise about it.

- **`score`**, from `-1.0` to `1.0`, is the **direction** of the emotion. Which way does the text lean.
- **`magnitude`**, from `0.0` upward with no ceiling, is the **amount** of emotion. Google's docs are explicit that it is not normalized: "each expression of emotion within the text (both positive and negative) contributes to the text's magnitude (so longer text blocks may have greater magnitudes)."

Because magnitude accumulates and score averages, the pair distinguishes two cases that a single number cannot:

| | score | magnitude | Reading |
| :--- | :--- | :--- | :--- |
| Clearly positive | 0.8 | 3.0 | Strong, consistently positive |
| Clearly negative | -0.6 | 4.0 | Strong, consistently negative |
| Neutral | 0.1 | 0.0 | Nothing much is being felt |
| **Mixed** | **0.0** | **4.0** | **A lot is being felt, in both directions** |

*(Sample values from Google's own [interpreting sentiment analysis values](https://cloud.google.com/natural-language/docs/basics) documentation.)*

Look at the last row. `score` near zero with a **high** magnitude is Google's way of expressing what AWS calls `MIXED`, and it is only visible if you keep both numbers. Throw magnitude away and mixed becomes indistinguishable from neutral.

Prove it to yourself.

In [ ]:
def gcp_analyze(text):
    """Return the full analyzeSentiment response for one text."""
    response = requests.post(
        "https://language.googleapis.com/v1/documents:analyzeSentiment",
        params={"key": os.environ["GCP_API_KEY"]},
        json={"document": {"type": "PLAIN_TEXT", "content": text}},
    )
    return response.json()


def gcp_sentiment(text):
    """Return (score, magnitude) from GCP Natural Language."""
    document_sentiment = gcp_analyze(text)["documentSentiment"]
    return document_sentiment["score"], document_sentiment["magnitude"]

Three probe texts, each about the same length, so length is not the thing driving the difference. Watch the **per sentence** scores, because that is where the mechanism is visible.

In [ ]:
probes = {
    "flat and factual": (
        "Requesting a copy of my declarations page. "
        "My mortgage company needs it by Friday. "
        "The policy number is on the renewal notice."
    ),
    "strongly positive": (
        "The adjuster was absolutely wonderful. "
        "She was incredibly kind and patient. "
        "The whole process was fast and easy. "
        "I could not be happier with how this was handled."
    ),
    "genuinely mixed": (
        "The adjuster was absolutely wonderful. "
        "She was incredibly kind and patient. "
        "But the payout was insulting and unfair. "
        "The three month delay was infuriating."
    ),
}

for name, text in probes.items():
    result = gcp_analyze(text)
    document = result["documentSentiment"]
    print(f"{name}:  score={document['score']:+.2f}  magnitude={document['magnitude']:.2f}")
    for sentence in result["sentences"]:
        print(f"    {sentence['sentiment']['score']:+.1f}  {sentence['text']['content']}")
    print()

### That is the whole mechanism, in one output

Read the per sentence scores next to the document totals.

- **Flat and factual** has near-zero sentence scores, so there is nothing to accumulate. Score near zero, magnitude near zero.
- **Strongly positive** has four sentences all pulling the same way. They reinforce, so score stays high **and** magnitude accumulates high.
- **Genuinely mixed** has two strongly positive sentences and two strongly negative ones. They **cancel in the score** and **add in the magnitude**. Score lands near zero. Magnitude lands as high as the strongly positive text, or higher.

Compare your two "score near zero" rows. The flat message and the mixed message have almost the same score, and wildly different magnitudes. If you had thrown magnitude away, those two messages would be indistinguishable, and one of them is a customer who is furious about half of their experience.

Your numbers should land close to the sample table above, which is a good sign that the table is describing real behavior rather than an idealized example.

Now go back to the message from Section 4, the one AWS called `POSITIVE` on a near coin flip.

In [ ]:
score, magnitude = gcp_sentiment(example_text)

print(f"text: {example_text}")
print()
print(f"AWS: {sentiment_response['Sentiment']}  "
      f"(top score {max(sentiment_response['SentimentScore'].values()):.3f})")
print(f"GCP: score={score:+.2f}  magnitude={magnitude:.2f}")

When this notebook was written the two providers landed on **opposite sides of neutral** for this message. AWS said positive by a hair. Google's score was negative.

Neither is wrong. Go back and read the message again: it is a short, contextless fragment with no strong emotional words, and reasonable humans would disagree about it too. That is what a near-tie in the AWS scores was trying to tell you in Section 4.

The lesson is not "one of these vendors is bad." It is that **on ambiguous input, the label you get depends on which vendor you bought**, and the only thing that warned you was a confidence score you had to go looking for.

---
# 10. The translation layer

AWS gives you `POSITIVE`. Google gives you `-0.3`. You cannot put those in the same column, count agreement, or hand either to a downstream job until they speak the same language.

So you write a mapping function. It will be about five lines, and it is the most consequential code in this notebook, because it introduces a number that **no API returned to you and no documentation will choose for you**: the threshold at which a score becomes a label.

Google says this outright:

> We recommend that you define a threshold that works for you, and then adjust the threshold after testing and verifying the results. For example, you may define a threshold of any score over 0.25 as clearly positive, and then modify the score threshold to 0.15 after reviewing your data.

Read that carefully, because it is unusually honest for vendor documentation. Google is telling you that the meaning of their output is **your** calibration problem, that the correct value depends on your data, and that you should expect to change it. `0.25` is an example in a doc, not a constant of nature.

Every cross-vendor pipeline is full of functions like this. They look trivial in a code review and they quietly determine the numbers everyone downstream trusts.

In [ ]:
# TODO: Write gcp_score_to_label(score, threshold=0.25).
#
#   score >  threshold  ->  "POSITIVE"
#   score < -threshold  ->  "NEGATIVE"
#   otherwise           ->  "NEUTRAL"
#
# Note what you CANNOT return: there is no "MIXED". A single score has no way
# to express it. You are structurally unable to reproduce one of AWS's four
# labels, and every MIXED row will therefore be counted as a disagreement no
# matter how well the models actually performed. Keep that in mind in Section 13
# before blaming the threshold for a number it did not cause.
#
# Make threshold a PARAMETER with a default, not a hardcoded number.
# Section 14 asks you to sweep it, and you do not want to be editing the
# function body to do that.

def gcp_score_to_label(score, threshold=0.25):
    pass

In [ ]:
# Sanity check the boundaries before trusting it on 60 rows.
for value in [-0.90, -0.30, -0.25, 0.00, 0.25, 0.30, 0.90]:
    print(f"{value:+.2f} -> {gcp_score_to_label(value)}")

Check the two boundary rows carefully. At exactly `+0.25` the function returns `NEUTRAL`, because the comparison is strictly greater than. Whether that is right is a judgment call, and the point of the check is that you now know which way it falls instead of finding out from a bug report.

---
# 11. Scale GCP across the same batch

Same 60 messages, same order, so the two results can be joined.

In [ ]:
# TODO: Run gcp_sentiment over every row in aws_df and build gcp_df.
#
# Iterate over aws_df (not posts) so the two frames cover exactly the same
# rows. If a message failed on the AWS side it is not in aws_df, and you do
# not want to pay GCP for a row you cannot compare.
#
# Columns: "textID", "gcp_score", "gcp_magnitude"
#
# Hint: gcp_sentiment(text) already returns the (score, magnitude) tuple.
# Hint: do NOT apply gcp_score_to_label here. Store the raw score and derive
#       the label in the next cell. Section 14 sweeps the threshold, and if
#       the label is baked in you have to re-run all 60 paid calls each time.
#       Store raw, derive cheap. This is a habit worth keeping.

gcp_rows = []

gcp_df = None

---
# 12. Compare, row by row

Join the two frames on `textID` and derive the GCP label from the stored score.

The shape you are building:

| textID | aws_sentiment | gcp_score | gcp_sentiment | agree |
| :--- | :--- | :--- | :--- | :--- |
| `910d626cd8` | POSITIVE | -0.30 | NEGATIVE | False |
| `8560ce3f2e` | NEUTRAL | +0.20 | NEUTRAL | True |

In [ ]:
# TODO: Merge aws_df and gcp_df on "textID" into `comparison`, add a
# "gcp_sentiment" column by applying gcp_score_to_label to "gcp_score",
# add a boolean "agree" column, and print the agreement rate.
#
# Hint: comparison["gcp_score"].apply(gcp_score_to_label)
# Hint: a boolean column's .mean() is the proportion True. Format with :.0%

comparison = None

When this notebook was written, agreement on these 60 social posts came out in the low sixties as a percentage. If you were expecting something in the nineties, that gap is the single most valuable result in the activity.

Two managed services, from the two largest cloud providers, both marketed as sentiment analysis, disagree on roughly **one message in three**.

In [ ]:
# TODO: Print the disagreements with the text, so you can judge them yourself.
#
# For each disagreeing row show: aws_sentiment, aws_confidence, gcp_score,
# gcp_sentiment, and the message text.
#
# Hint: comparison[~comparison["agree"]]
# Hint: sort by aws_confidence ascending, so the rows where AWS was least sure
#       come first. Those are usually the ones that are genuinely ambiguous
#       rather than a real vendor difference.

### Sort the disagreements into three piles

Do not skim these. Read them and classify each one, because the three causes have three completely different fixes:

1. **Genuinely ambiguous text.** Short fragments, replies with no context, sarcasm. Two humans would disagree too. No threshold and no vendor fixes this. The fix is human review, or accepting the error rate.
2. **A threshold artifact.** GCP scored `-0.28` and your cutoff was `0.25`, so a barely-negative message became `NEGATIVE` while AWS called it `NEUTRAL`. Nothing disagreed about the *content*, only about where you drew a line. Section 14 measures how many of these there are.
3. **A real model difference.** One provider clearly read the message correctly and the other clearly did not. These are the only ones that are actually evidence about vendor quality.

Reporting a bare agreement number without doing this sort is how people end up making a six-figure vendor decision on top of an artifact of their own default parameter.

---
# 13. How much of that was MIXED?

Before blaming the threshold, account for the disagreements you made structurally impossible in Section 10.

Every row AWS labeled `MIXED` is guaranteed to disagree, because `gcp_score_to_label` cannot return `MIXED`. Those rows are not measuring model quality at all. They are measuring the fact that the two vendors chose different label vocabularies.

In [ ]:
# TODO: Count how many disagreements are AWS MIXED rows, and recompute the
# agreement rate over only the rows where AWS did NOT say MIXED.
#
# Hint: comparison["aws_sentiment"] == "MIXED"
# Hint: comparison[~is_mixed]["agree"].mean()

The excluded-MIXED number should be a little higher, but only a little. The vocabulary mismatch is real and worth reporting, and it is not what is driving the disagreement. Most of the gap is genuine.

This is the honest way to report a comparison: state the number, then account for the part of it that is an artifact of your own decisions, then state what is left. A stakeholder who later discovers the MIXED problem on their own will discount everything else you told them.

---
# 14. Sweep the threshold

You picked `0.25` because Google's documentation used it as an example. That is a starting point, not a justification. Find out what it actually costs you.

Because you stored the raw scores in Section 11, this sweep costs zero API calls. That was the point of storing raw and deriving cheap.

In [ ]:
# TODO: For each threshold in [0.05, 0.10, 0.15, 0.25, 0.35, 0.50],
# recompute the GCP labels and the agreement rate, and also show how the
# GCP label distribution shifts.
#
# Hint: comparison["gcp_score"].apply(lambda s: gcp_score_to_label(s, threshold))
# Hint: build a list of dicts and turn it into a DataFrame so the sweep reads
#       as a table rather than a wall of prints.

### Read the sweep carefully, it does not say what people expect

Two things should jump out.

**Agreement does not climb steadily as you tune.** It moves within a narrow band across the low and middle thresholds and then falls off at the high end. There is no hidden setting that makes these two providers agree. You are not one parameter away from a clean answer, and that is worth knowing before you spend a day looking for it.

**The label distribution changes a lot even where agreement barely moves.** As the threshold rises, `NEUTRAL` absorbs more and more rows, until at the high end most messages are classified as having no sentiment at all. If your dashboard reports "percent of customers unhappy this week," that number is partly a function of a constant somebody typed once and nobody revisited.

So what is the threshold actually for? Not maximizing agreement with a competitor. It is for calibrating against **your** definition of positive and negative on **your** data, which requires labels you do not have today. That is the honest limit of this exercise, and it is why the next section is about a different corpus rather than a better number.

---
# 15. Same code, different corpus

The pipeline is done. Now run it unchanged on a different kind of text and watch the answer change.

`claims_feedback_sample.csv` is 18 customer and claims messages: phone transcripts, emails, and portal submissions. Compared to social posts they are longer, better punctuated, written to be understood, and about a single topic each.

Before you run it, predict: will agreement go up or down, and by how much? Write the number down.

In [ ]:
# TODO: Run the whole pipeline over `claims`, exactly as you built it,
# and report the agreement rate.
#
# The only things that change are the column names: the id column is
# "message_id" instead of "textID".
#
# Build claims_comparison with at least:
#   message_id | channel | aws_sentiment | aws_confidence | gcp_score |
#   gcp_sentiment | agree | text
#
# Hint: this is 18 messages against each provider, 36 calls total.
# Hint: if you find yourself copying and pasting the Section 8 and Section 11
#       loops, that is a signal. Consider wrapping the per-message work in a
#       function like classify(text) -> dict and calling it from both places.
#       Activity 7 turns exactly this into a real pipeline.

claims_comparison = None

### The most important result in the notebook

Agreement on the claims messages should come out substantially **higher** than on the social posts. When this notebook was written it was in the low eighties against the low sixties, using the identical threshold and the identical code.

**Nothing about the models changed. Only the text changed.**

That has a hard consequence for how you evaluate any AI service:

> A benchmark number is a property of a model **and a dataset**, never of a model alone.

A vendor's marketing page reporting 92 percent accuracy is reporting it on text of the vendor's choosing. Your text is not that text. The only number that tells you anything about your workload is the one you measured on your workload, which is why this activity had you run both providers on your own data rather than read a comparison chart.

It also tells you something concrete about scoping work at Hartford Mutual. Well-formed business writing is the easy case, and both vendors handle it comparably, so for claims correspondence the choice between them can be made on price, latency, and what else is in the ecosystem. Short, informal, contextless text is the hard case, and there the vendor choice materially changes your numbers.

In [ ]:
# TODO: MSG_013 is worth arguing about. Print it along with both providers'
# verdicts, then decide what the "right" answer even is.
#
# Hint: claims_comparison[claims_comparison["message_id"] == "MSG_013"]

Read that message. Someone's car was stolen and they are filing a claim.

The text is calm and factual. It reports a police report number. It contains no emotional language at all. When this notebook was written, AWS scored it as confidently `NEUTRAL`, and Google returned a strongly negative score.

**Which one is correct?** Argue it out before reading on.

The case for `NEUTRAL`: sentiment analysis measures the emotion expressed *in the text*, and this text expresses none. The customer wrote it in the register of a police report. AWS is measuring exactly what it claims to measure.

The case for negative: the *event* is unambiguously bad, and any system routing these messages should treat "my car was stolen" as urgent. Google's score is more useful even if it is arguably answering a different question.

**They are both right, about different questions.** And that is the actual finding: "sentiment" is not one well-defined thing. If your requirement is "flag unhappy customers," you want emotional tone. If your requirement is "route urgent claims," sentiment is the wrong tool entirely and you should be classifying claim type and severity instead.

A model can only tell you the answer to the question it was trained on. Deciding whether that is your question is your job, and it happens before you write any code.

---
# 16. The deliverable

Same as Activity 1. The output of this notebook is not printed cells, it is a file somebody else can use.

In [ ]:
# TODO: Write both comparison tables to CSV, then read one back to confirm.
#
#   comparison         -> sentiment_social_posts.csv
#   claims_comparison  -> sentiment_claims.csv
#
# Hint: index=False, for the same reason as Activity 1.
# Hint: keep the raw gcp_score and the aws_* score columns in the file.
#       Whoever picks this up next may want to re-derive labels at a different
#       threshold, and if you only saved the labels they have to pay for
#       every call again to do it.

In [ ]:
claims_comparison.groupby("channel")["agree"].agg(["mean", "count"])

---
# 17. Why not just ask an LLM?

Fair question, and you could. Day 3 did something close to it.

Here is why a data engineer reaches for a managed NLP service some or all of the time:

- **Stable.** The same input returns the same label. An LLM at any temperature above zero can drift, and even at temperature zero the model behind the endpoint gets updated.
- **No hallucination surface.** This is classification against a fixed label set. The service cannot invent a fifth sentiment.
- **Cheaper at volume.** A managed NLP call is typically a fraction of the cost of a chat completion on the same input, and at 400,000 rows that ratio is the entire budget conversation.
- **Nothing to maintain.** No prompt, no output parser, no schema validation, no retry-on-malformed-JSON. Your sentiment numbers do not silently shift because a teammate improved a system prompt.

The trade is flexibility. Comprehend will never summarize the message, answer a follow-up, or reason about whether the claim looks fraudulent. It does one narrow thing.

And notice what this activity established that cuts the other way: managed services are not the safe, boring, obviously-correct choice either. Two of them disagreed on a third of your messages, one of them cannot express a label the other one returns, and the number you report depends on a threshold you invented. "Managed" removes the model risk. It does not remove the judgment.

That is exactly the territory the afternoon's [group activity](./Group_Activity_AI_Decision_Hierarchy.md) asks you to reason about.

---
# Your Turn

Work in your own copy under `student-work/week6/day4/`.

### 1. Label 20 messages yourself (required)

You have been measuring agreement all activity, because there is no ground truth in either file. Make some.

Take 20 messages from `comparison`, label them yourself as POSITIVE, NEGATIVE, or NEUTRAL **before** looking at either provider's answer, then compute each provider's **accuracy** against your labels.

Report three numbers: AWS accuracy, GCP accuracy, and the agreement rate you already had. Then answer the question the whole activity has been circling: was the agreement rate a useful proxy for accuracy, or was it misleading?

### 2. Redact the PII (required)

`detect_pii_entities` was on the menu in Section 1 and you have not used it. Run it over `claims_feedback_sample.csv` and find every message containing personally identifiable information.

Then use the `BeginOffset` and `EndOffset` values to write a `redact(text)` function that replaces each PII span with its entity type, so `police report number is HPD-88291` becomes `police report number is [OTHER]` or similar.

Work backwards through the entity list when you replace, from the last offset to the first. Replacing forwards shifts every offset after the one you just changed, and the result will be quietly wrong in a way that is hard to spot. Explain in a comment why that is.

### 3. Does channel matter? (required)

The last cell of Section 16 groups agreement by `channel`. Phone transcripts read very differently from typed portal submissions.

With only 18 messages the per-channel counts are tiny, so the honest answer may be "not enough data to say." Deciding that a difference is not yet meaningful is a real analytical result. State how many messages per channel you would want before you would report the comparison to anyone.

### Stretch goals

- Rewrite the Section 8 loop with `batch_detect_sentiment` in chunks of 25. Time both versions. Then handle the `ErrorList` correctly by joining on `Index`, and construct a test case that proves your version does not misalign rows when one document fails.
- Run `detect_targeted_sentiment` on a claims message that praises one thing and criticizes another. Does entity-level sentiment capture what document-level sentiment flattened out?
- Take the three most ambiguous messages by `aws_confidence` and ask an LLM to classify them, with a prompt that requires a one-sentence justification. Read the justifications. Does the reasoning change your view of which label is right, and what does that suggest about where a managed classifier ends and an LLM begins?

---
## What you did

- Read a managed service's full operation menu off the client object instead of assuming it does one thing.
- Navigated an unfamiliar response dictionary and found a near coin flip hiding under a confident-looking label.
- Learned what `MIXED` is and predicted, correctly, that it would break the comparison before it did.
- Used the documentation to find size limits, the exact label vocabulary, and a batch operation you were not using.
- Built a translation layer between two incompatible vendor outputs, and named the judgment call inside it.
- Swept that judgment call across six values and found that no setting rescues the comparison.
- Ran identical code on two corpora and got 20 points of difference, from the text alone.
- Wrote out structured datasets that keep the raw scores, so the next person can re-derive labels without paying for the calls again.

**Next:** [Activity 3](./Activity_3_Keyword_to_Semantic_Search.ipynb) moves from classifying one document at a time to searching across many, and finds the exact point where keyword matching fails.